# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ashu141d/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [21]:
!cat /content/repo/skills/README.md
!echo "=== DATA SKILL ==="
!cat /content/repo/skills/flyrank/flyrank-data/SKILL.md 2>/dev/null || find /content/repo/skills -name "*.md" -exec head -n 120 {} +

# Skills — the router

This folder is a small library of **skills**: focused instruction files your AI assistant loads
one at a time. One skill per task keeps the assistant sharp — its context window is small, and
filling it with everything makes it worse at the one thing you need.

**How to use it (repo-reading agents — Claude Code, Cursor, Codex):** they find this file
automatically via `AGENTS.md` / `CLAUDE.md`. Just tell your assistant which task you're doing.

**Using a chat-only assistant (ChatGPT / Gemini in a browser)?** Open the skill file on GitHub,
copy its whole content, and paste it into your chat before asking for help. That's it.

## The table — find your task, load ONE skill

| Your task | Load this skill | Also load for data work |
|---|---|---|
| Any task — how to work with your assistant at all | `directing-your-ai-assistant/SKILL.md` | — |
| Pick a lane, frame your question (ML-02, ML-03) | `framing-ml-problems/SKILL.md` | `flyrank/flyrank-data/SKILL.md` |
| Write +

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [22]:
import os
import itertools
import pandas as pd
from google.colab import userdata
from datasets import load_dataset

# 1. Authenticate using Colab Secret HF_TOKEN (ensure Name='HF_TOKEN' and Value is pasted)
hf_token = userdata.get("HF_TOKEN")
os.environ["HF_TOKEN"] = hf_token

# 2. Stream a slice of fact_content_daily_performance
ds = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    streaming=True,
    split="train",
    token=hf_token
)

# Pull first 2,000 rows into a DataFrame for fast inspection & contract checks
raw_df = pd.DataFrame(list(itertools.islice(ds, 2000)))
print(f"Loaded streamed sample shape: {raw_df.shape}")
display(pd.DataFrame({"column": raw_df.columns, "dtype": raw_df.dtypes.astype(str).values}))
display(raw_df.head(3))

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Loaded streamed sample shape: (2000, 30)


,column,dtype
0,report_date,object
1,client_hash_id,object
2,content_hash_id,object
3,client_has_gsc,bool
4,client_has_ga4,bool
5,gsc_data_available,bool
6,ga4_data_available,bool
7,gsc_impressions,int64
8,gsc_clicks,int64
9,gsc_sum_position,int64


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,...,0,0,0,0,0,0,0,0,0,0
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358,...,0,0,0,0,0,0,0,0,0,0
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34,...,0,0,0,0,0,0,0,0,0,0


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [23]:
cols = raw_df.columns.tolist()
date_col = next((c for c in cols if "date" in c.lower() or "month" in c.lower() or "day" in c.lower()), cols[0])
avail_col = next((c for c in cols if "avail" in c.lower() or "valid" in c.lower() or "eligible" in c.lower()), None)

raw_df["_period_str"] = raw_df[date_col].astype(str)
raw_df["_month_str"] = raw_df["_period_str"].str.slice(0, 7)

print(f"Detected date column: '{date_col}' | Availability column: '{avail_col}'")
print(f"Sample row count: {len(raw_df)} | Date span in slice: {raw_df['_period_str'].min()} -> {raw_df['_period_str'].max()}")
print("Months present in slice:", sorted(raw_df["_month_str"].unique().tolist()))

Detected date column: 'report_date' | Availability column: 'gsc_data_available'
Sample row count: 2000 | Date span in slice: 2025-01-27 -> 2025-02-10
Months present in slice: ['2025-01', '2025-02']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [24]:
# Identify key column and up to 5 numeric feature columns dynamically
key_col = next((c for c in cols if any(k in c.lower() for k in ["page", "content", "url", "id", "slug"]) and c != date_col), cols[0])
num_cols = [c for c in raw_df.select_dtypes(include="number").columns if c not in [date_col, key_col]][:5]

# Select mid-panel month ('2026-03' if available in streamed slice, else middle month)
months = sorted(raw_df["_month_str"].unique().tolist())
target_month = "2026-03" if "2026-03" in months else months[len(months) // 2]
mid_df = raw_df[raw_df["_month_str"] == target_month].copy()

# QUERY 1: Grain check on mid-panel month
grain_check = pd.DataFrame([{
    "target_month": target_month,
    "total_rows": len(mid_df),
    "distinct_entity_date_pairs": mid_df.drop_duplicates(subset=[key_col, date_col]).shape[0],
    "duplicate_rows": len(mid_df) - mid_df.drop_duplicates(subset=[key_col, date_col]).shape[0]
}])
print("--- Check 1: Grain Uniqueness ---")
display(grain_check)

# QUERY 2: Availability filter (IS TRUE) survival check
if avail_col:
    surviving_df = mid_df[mid_df[avail_col] == True].copy()
else:
    surviving_df = mid_df.dropna(subset=num_cols[:1]).copy()

avail_check = pd.DataFrame([{
    "target_month": target_month,
    "rows_before_filter": len(mid_df),
    "rows_is_true": len(surviving_df),
    "survival_pct": round(100.0 * len(surviving_df) / max(len(mid_df), 1), 2)
}])
print("\n--- Check 2: Availability Filter (IS TRUE) ---")
display(avail_check)

# QUERY 3: 5-Feature Frame + Intentional Leakage Trap Demonstration
feature_cols = [key_col, date_col] + num_cols
df_features = surviving_df[feature_cols].head(500).copy()
print("\n--- Check 3: Clean 5-Feature Frame (Known at Decision Time t) ---")
display(df_features.head())

if len(num_cols) >= 1 and len(df_features.dropna()) > 10:
    clean_sub = df_features.dropna().copy()
    primary_metric = num_cols[0]
    y_target = (clean_sub[primary_metric] < clean_sub[primary_metric].median()).astype(int)

    # Deliberately add a label-derived leaky column
    leaky_sub = clean_sub.copy()
    leaky_sub["LEAKY_future_outcome_proxy"] = y_target * 0.98 + 0.01

    print("\nLeakage Trap Experiment:")
    print(f"  Honest feature ({primary_metric}) corr : {clean_sub[primary_metric].corr(y_target):.4f}")
    print(f"  LEAKY column corr                     : {leaky_sub['LEAKY_future_outcome_proxy'].corr(y_target):.4f}  <-- Artificially near-perfect!")

    # Delete the leaky column and keep the honest frame
    df_features = leaky_sub.drop(columns=["LEAKY_future_outcome_proxy"])
    print("Leaky column removed. Honest feature frame kept.")

--- Check 1: Grain Uniqueness ---


,target_month,total_rows,distinct_entity_date_pairs,duplicate_rows
0,2025-02,703,1,702



--- Check 2: Availability Filter (IS TRUE) ---


,target_month,rows_before_filter,rows_is_true,survival_pct
0,2025-02,703,703,100.0



--- Check 3: Clean 5-Feature Frame (Known at Decision Time t) ---


,client_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_sum_position,gsc_avg_position,ga4_pageviews
1297,client_9958f0a7ae1df715,2025-02-10,9,0,230,25.555556,0
1298,client_9958f0a7ae1df715,2025-02-10,5,0,44,8.800000,0
1299,client_9958f0a7ae1df715,2025-02-10,13,0,557,42.846154,0
1300,client_9958f0a7ae1df715,2025-02-10,27,0,922,34.148148,0
1301,client_9958f0a7ae1df715,2025-02-10,6,0,75,12.500000,0



Leakage Trap Experiment:
  Honest feature (gsc_impressions) corr : -0.5316
  LEAKY column corr                     : 1.0000  <-- Artificially near-perfect!
Leaky column removed. Honest feature frame kept.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [25]:
null_report = pd.DataFrame({
    "feature": df_features.columns,
    "non_null_count": df_features.notnull().sum().values,
    "null_count": df_features.isnull().sum().values,
    "null_pct": (df_features.isnull().mean() * 100).round(2).values
})
print("Missingness & Data Limits Check:")
display(null_report)

Missingness & Data Limits Check:


,feature,non_null_count,null_count,null_pct
0,client_hash_id,500,0,0.0
1,report_date,500,0,0.0
2,gsc_impressions,500,0,0.0
3,gsc_clicks,500,0,0.0
4,gsc_sum_position,500,0,0.0
5,gsc_avg_position,500,0,0.0
6,ga4_pageviews,500,0,0.0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.